# Task 12, Notebook 05: Twin-resume audit, baseline

**Programme:** Aura Workforce by Growmar, AI/ML track, CortexHR **Phase 3: Intelligence**  
**Task:** 12: Audit your screener for fairness  
**Objective:** Validate the 25 twin pairs, measure the score gap of the screener as first built, and find out where any gap comes from.  
**Workflow stages covered in this notebook:** 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 24 of the 38-stage Phase 1/2 standard.  
**Rule for every number below:** it is printed by the code in the same notebook; nothing is typed in by hand.

In [1]:
import sys, warnings, pathlib
warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'config' / 'phase3_config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
from cortexhr.phase3.common.config import load_config
from cortexhr.phase3.common import paths as P
CFG = load_config(); SEED = CFG['seed']
print('project root:', ROOT.name, '| seed:', SEED)

project root: phase3_working | seed: 42


## Stage 1: Business Understanding

*What this stage is for:* Who needs this result and what decision it supports.

A screener that scores a resume differently because of a name has failed before the first interview. Companies need a *documented* fairness test before an HR model goes live. This notebook is that test for the Task 11 screener.

## Stage 2: Problem Definition

*What this stage is for:* The exact prediction, ranking or measurement problem, its inputs and its output.

**Counterfactual test.** Take 25 pairs of resumes that are identical except for one detail (a gender-coded name, sometimes also the college tier). Score both. If the only difference is the protected detail, any score difference is unfair treatment. **Output:** the gap per pair, the average gap, and where it comes from.

## Stage 3: Objective and Success Criteria

*What this stage is for:* What number or evidence shows the task is done well, decided before looking at results.

Pass mark, fixed in the config before measuring: the average and maximum twin gap after the fix must be at most the tolerance, and ranking quality must not fall by more than the allowed amount.

In [2]:
from cortexhr.phase3.common.io import read_csv, save_table, save_json
from cortexhr.phase3.common.validation import profile
from cortexhr.phase3.resume import text as T
from cortexhr.phase3.resume.scoring import ResumeRanker
from cortexhr.phase3.resume.metrics import attach_labels, evaluate
from cortexhr.phase3.fairness.twins import validate_pairs, twin_gaps, gap_summary, attach_protected, vault_resume_attributes
resumes = read_csv('resumes'); jobs = read_csv('job_descriptions'); labels = read_csv('resume_job_labels'); pairs = read_csv('resume_bias_pairs'); dd = read_csv('data_dictionary')
attrs = vault_resume_attributes()   # restricted file: used ONLY to interpret results after scoring; None if the file is not present
print('twin pairs:', len(pairs), '| restricted attribute file available:', attrs is not None)

twin pairs: 25 | restricted attribute file available: True


In [3]:
pd.Series({'tolerance for a practical gap': CFG['fairness']['tolerance'], **{k: v for k, v in CFG['resume']['success_criteria'].items() if k in ('max_twin_gap_after_fix', 'max_ndcg_drop_after_fix')}}, name='value').to_frame()

                               value
tolerance for a practical gap  0.001
max_twin_gap_after_fix         0.001
max_ndcg_drop_after_fix        0.050

## Stage 4: Dataset Discovery

*What this stage is for:* Which files exist, how they relate, and which are authoritative.

Files: `resume_bias_pairs` (25 pairs), `resumes` (both twins are in it), `resume_job_labels` (the twins were also scored by humans on 3 jobs each), `job_descriptions`, and the restricted `unlock_CX-12` file with the hidden protected attributes. The restricted file is never an input to any score.

## Stage 5: Data Loading

*What this stage is for:* Load with the project helpers and check the load is complete.

In [4]:
{'pairs': pairs.shape, 'resumes': resumes.shape, 'labels': labels.shape, 'twin resumes labelled': int(labels.resume_id.isin(pairs.resume_id_twin).sum())}

{'pairs': (25, 8), 'resumes': (185, 15), 'labels': (555, 7), 'twin resumes labelled': 75}

## Stage 6: Dataset Structure and Shape

*What this stage is for:* Rows, columns, keys and grain of each table.

One row per pair; each pair gives two resumes and, through the labels, up to 3 jobs, so 25 pairs give 75 comparisons.

In [5]:
pairs.head(3)

  pair_id resume_id_original resume_id_twin                               attribute_changed skills_identical experience_identical  expected_score_difference                                            instruction
0   BP-01           RES-0123       RES-0161  candidate_name (gender-coded) and college tier                Y                    Y                          0  A fair screener must give both resumes the same score
1   BP-02           RES-0114       RES-0162              candidate_name (gender-coded) only                Y                    Y                          0  A fair screener must give both resumes the same score
2   BP-03           RES-0005       RES-0163  candidate_name (gender-coded) and college tier                Y                    Y                          0  A fair screener must give both resumes the same score

## Stage 7: Data Dictionary and Schema Understanding

*What this stage is for:* What each column means, from the supplied dictionary and from inspection.

In [6]:
dd[dd.sheet.str.contains('bias', case=False)][['sheet', 'column', 'data_type', 'description']]

                 sheet                     column data_type                                         description
224  resume_bias_pairs                    pair_id      text                                       Bias pair id.
225  resume_bias_pairs         resume_id_original      text                                    Original resume.
226  resume_bias_pairs             resume_id_twin      text                                        Twin resume.
227  resume_bias_pairs          attribute_changed      text             What differs between original and twin.
228  resume_bias_pairs           skills_identical      text                             Y: skills are the same.
229  resume_bias_pairs       experience_identical      text                          Y: experience is the same.
230  resume_bias_pairs  expected_score_difference   integer  Expected score difference (0 for a fair screener).
231  resume_bias_pairs                instruction      text                                       What t

## Stage 8: Data Type Inspection

*What this stage is for:* Check every column has the type it should have.

In [7]:
pairs.dtypes.astype(str).to_frame('dtype')

                            dtype
pair_id                    object
resume_id_original         object
resume_id_twin             object
attribute_changed          object
skills_identical           object
experience_identical       object
expected_score_difference   int64
instruction                object

## Stage 9: Data Quality Assessment

*What this stage is for:* A one-page quality profile before any modelling.

In [8]:
profile(pairs)

                            dtype  missing  missing_pct  unique
pair_id                    object        0          0.0      25
resume_id_original         object        0          0.0      25
resume_id_twin             object        0          0.0      25
attribute_changed          object        0          0.0       2
skills_identical           object        0          0.0       1
experience_identical       object        0          0.0       1
expected_score_difference   int64        0          0.0       1
instruction                object        0          0.0       1

## Stage 10: Missing Value Analysis

*What this stage is for:* Where values are missing, how much, and why that may be.

In [9]:
print('missing values in pairs:', pairs.isna().sum().sum(), '| twin ids found in resumes:', pairs.resume_id_twin.isin(resumes.resume_id).all(), pairs.resume_id_original.isin(resumes.resume_id).all())

missing values in pairs: 0 | twin ids found in resumes: True True


## Stage 11: Duplicate Analysis

*What this stage is for:* Exact and near duplicates of rows, keys and texts.

A resume must belong to at most one pair, and a pair must not be its own twin.

In [10]:
ids = pd.concat([pairs.resume_id_original, pairs.resume_id_twin]); print('resume ids used in pairs:', len(ids), '| unique:', ids.nunique(), '| pair_id unique:', pairs.pair_id.is_unique, '| original == twin anywhere:', bool((pairs.resume_id_original == pairs.resume_id_twin).any()))

resume ids used in pairs: 50 | unique: 50 | pair_id unique: True | original == twin anywhere: False


## Stage 12: Outlier Detection and Investigation

*What this stage is for:* Unusual values: are they errors or real?

For this table an outlier means an unusual pair: one where more than the expected details differ. Measured as how many characters of the resume text differ.

In [11]:
import difflib
r = resumes.set_index('resume_id'); size = []
for _, p in pairs.iterrows():
    a, b = r.loc[p.resume_id_original, 'resume_text'], r.loc[p.resume_id_twin, 'resume_text']
    size.append(sum(len(x) for x in [a, b]) - 2 * sum(m.size for m in difflib.SequenceMatcher(None, a, b, autojunk=False).get_matching_blocks()))
pairs['changed_characters'] = size; pairs['changed_characters'].describe().round(1).to_frame().T

                    count  mean  std   min   25%   50%   75%   max
changed_characters   25.0  37.4  8.9  19.0  34.0  40.0  42.0  52.0

## Stage 13: Invalid and Inconsistent Value Detection

*What this stage is for:* Impossible values, broken links between tables, label contradictions.

Are the twins really identical apart from the expected details? This check compares every column.

In [12]:
v = validate_pairs(pairs, resumes); print('pairs with an unexpected difference:', int((v.unexpected_differences != '').sum()), '| skills identical in all pairs:', bool(v.skills_actually_identical.all()), '| experience identical in all pairs:', bool(v.experience_identical.all()))
v.groupby(['attribute_changed', 'columns_that_differ']).size().to_frame('pairs')

pairs with an unexpected difference: 0 | skills identical in all pairs: True | experience identical in all pairs: True


                                                                                                 pairs
attribute_changed                              columns_that_differ                                    
candidate_name (gender-coded) and college tier candidate_name, email, college_tier, resume_text     13
candidate_name (gender-coded) only             candidate_name, email, resume_text                   12

## Stage 14: Exploratory Data Analysis (EDA)

*What this stage is for:* Look at the data before modelling.

In [13]:
pairs.attribute_changed.value_counts().to_frame('pairs')

                                                pairs
attribute_changed                                    
candidate_name (gender-coded) and college tier     13
candidate_name (gender-coded) only                 12

## Stage 15: Univariate Analysis

*What this stage is for:* One variable at a time.

In [14]:
pairs.expected_score_difference.value_counts().to_frame('pairs'); 

                           pairs
expected_score_difference       
0                             25

## Stage 16: Bivariate Analysis

*What this stage is for:* Two variables together, especially against the target or label.

Which details change in which pairs, using the restricted attribute file (interpretation only).

In [15]:
if attrs is None: print('restricted file not present: skipped')
else:
    a = attrs; flip = pd.DataFrame({'gender_original': a.loc[pairs.resume_id_original, 'inferred_gender'].values, 'gender_twin': a.loc[pairs.resume_id_twin, 'inferred_gender'].values,
                                   'tier_original': a.loc[pairs.resume_id_original, 'college_tier'].values, 'tier_twin': a.loc[pairs.resume_id_twin, 'college_tier'].values})
    print('pairs where the gender-coded name flips:', int((flip.gender_original != flip.gender_twin).sum()), 'of', len(flip), '| pairs where college tier also changes:', int((flip.tier_original != flip.tier_twin).sum()))
    print(pd.crosstab(flip.gender_original, flip.gender_twin))

pairs where the gender-coded name flips: 25 of 25 | pairs where college tier also changes: 13
gender_twin      Female  Male
gender_original              
Female                0    14
Male                 11     0


## Stage 17: Multivariate Analysis

*What this stage is for:* Several variables together: correlation, interaction, redundancy.

In [16]:
if attrs is not None:
    print(pd.crosstab([flip.gender_original + '->' + flip.gender_twin], flip.tier_original + '->' + flip.tier_twin).to_string())
else: print('skipped (restricted file not present)')

col_0         Tier 1->Tier 1  Tier 1->Tier 3  Tier 2->Tier 1  Tier 2->Tier 2  Tier 3->Tier 1  Tier 3->Tier 3
row_0                                                                                                       
Female->Male               1               1               3               4               4               1
Male->Female               3               1               1               2               3               1


## Stage 18: Target or Label Analysis

*What this stage is for:* How the target or human label is distributed and how reliable it is.

The 'label' for a fairness test is the *expected* gap, which is zero for every pair. A second reference: did the **human reviewers** score twins differently?

In [17]:
hl = labels.set_index(['resume_id', 'job_id']).human_suitability_score; rows = []
for _, p in pairs.iterrows():
    for job in sorted(set(labels[labels.resume_id == p.resume_id_original].job_id) & set(labels[labels.resume_id == p.resume_id_twin].job_id)):
        rows.append({'pair_id': p.pair_id, 'job_id': job, 'human_diff_twin_minus_original': hl[(p.resume_id_twin, job)] - hl[(p.resume_id_original, job)]})
hd = pd.DataFrame(rows); print('expected gap in the pairs file:', pairs.expected_score_difference.unique().tolist(), '| human score differences between twins:', hd.human_diff_twin_minus_original.value_counts().to_dict())

expected gap in the pairs file: [0] | human score differences between twins: {0: 75}


## Stage 19: Data Leakage Investigation

*What this stage is for:* Anything that carries the answer or protected information into the model.

Where can a name or a college tier enter a text-based score? Compare the words of the two twins.

In [18]:
o, t = pairs.iloc[0].resume_id_original, pairs.iloc[0].resume_id_twin
to, tt = set(T.tokenize(r.loc[o, 'resume_text'])), set(T.tokenize(r.loc[t, 'resume_text']))
print('pair', pairs.iloc[0].pair_id, '| words only in the original:', sorted(to - tt)); print('words only in the twin:', sorted(tt - to))
diffs = []
for _, p in pairs.iterrows():
    a_, b_ = set(T.tokenize(r.loc[p.resume_id_original, 'resume_text'])), set(T.tokenize(r.loc[p.resume_id_twin, 'resume_text'])); diffs.append(len(a_ ^ b_))
print('differing word types per pair: min %d, median %d, max %d' % (min(diffs), np.median(diffs), max(diffs)))

pair BP-01 | words only in the original: ['banerjee', 'pavithra', 'pavithra.banerjee']
words only in the twin: ['aditya', 'aditya.krishnan', 'krishnan']
differing word types per pair: min 4, median 6, max 6


**Leakage finding.** The twins differ in the name, the e-mail (built from the name) and, in some pairs, the college-tier phrase. A TF-IDF model reading `resume_text` as given sees those words. Excluding the *columns* `candidate_name` and `college_tier` is not enough, because the same information sits in the free text.

## Stage 24: Baseline Establishment

*What this stage is for:* The simple method every model must beat.

The baseline for fairness is **the screener as first built** (v1): the Task 11 engine with `resume_text` used as given and no masking. Its gap is the number to beat.

In [19]:
v1 = ResumeRanker(mask=False).fit(resumes, jobs)    # the screener as first built: resume_text exactly as given
v2 = ResumeRanker(mask=True).fit(resumes, jobs)     # after the fix: identifiers masked
def labelled_eval(r): return evaluate(attach_labels(r.score_all(labels[['resume_id', 'job_id']]), labels), 5)

In [20]:
g1 = twin_gaps(v1, pairs, labels); s1 = gap_summary(g1); pd.Series(s1, name='v1 (before the fix)').to_frame()

                                v1 (before the fix)
comparisons                               75.000000
pairs                                     25.000000
mean_abs_gap                               0.000126
median_abs_gap                             0.000028
max_abs_gap                                0.001619
mean_signed_gap(twin-original)            -0.000031
comparisons_with_nonzero_gap              74.000000
comparisons_above_tolerance                2.000000
mean_abs_relative_gap_pct                  0.069062
share_twin_ranked_higher                   0.480000
mean_abs_rank_difference                   1.000000

In [21]:
print('Interpretation: the gap is defined as twin score minus original score (score scale 0 to 1, so 0.001 = 0.1 score points). Mean absolute gap %.6f, maximum %.6f, non-zero in %d of %d comparisons.' % (s1['mean_abs_gap'], s1['max_abs_gap'], s1['comparisons_with_nonzero_gap'], s1['comparisons']))
print('Mean signed gap %.6f: positive would mean twins score higher on average; near zero with mixed signs means no systematic favouring.' % s1['mean_signed_gap(twin-original)'])

Interpretation: the gap is defined as twin score minus original score (score scale 0 to 1, so 0.001 = 0.1 score points). Mean absolute gap 0.000126, maximum 0.001619, non-zero in 74 of 75 comparisons.
Mean signed gap -0.000031: positive would mean twins score higher on average; near zero with mixed signs means no systematic favouring.
